In [203]:
import sys
import pathlib
import numpy as np
import torch
import torch.nn.functional as F
import scipy.stats as stats
from hadamard_transform import randomized_hadamard_transform, inverse_randomized_hadamard_transform, pad_to_power_of_2

PROJECT_PATH = pathlib.Path.cwd().parent
if str(PROJECT_PATH) not in sys.path:
    sys.path.append(str(PROJECT_PATH))

from pcdvq import (
    e8_minimal_directions,
    construct_direction_codebook,
    construct_magnitude_codebook,
    PCDVQ,
)
from pcdvq.utils import reshape_pq_to_k, reshape_k_to_pq
from pcdvq.standard_regularization import RandomizedHadamard

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [204]:
tensors_dumps_dir = "tensors_dumps"
weight_filename = "weight.pt"
weight_path = PROJECT_PATH / tensors_dumps_dir / weight_filename
weight = torch.load(weight_path)
p, q  = weight.shape
weight

tensor([[-2.1744e-04, -1.0193e-02,  9.5825e-03,  ..., -3.1738e-03,
         -8.6670e-03,  1.3504e-03],
        [ 2.8076e-02, -2.3804e-03, -1.2634e-02,  ..., -1.9989e-03,
          1.9684e-03,  1.5503e-02],
        [-1.6357e-02,  5.8289e-03, -1.7456e-02,  ..., -1.1414e-02,
          1.6174e-03,  2.8372e-05],
        ...,
        [ 2.2949e-02, -1.2024e-02, -2.3438e-02,  ..., -2.0752e-02,
          3.9795e-02,  2.7588e-02],
        [-2.9663e-02,  2.0142e-02,  1.4465e-02,  ..., -2.7222e-02,
         -6.2561e-03, -3.8818e-02],
        [ 1.1169e-02,  4.4922e-02, -7.2937e-03,  ...,  3.0518e-02,
         -2.3438e-02, -2.3071e-02]], dtype=torch.float16)

In [207]:
phi_bits = 6
r_bits = 2
k = 64
tau = 0.15
tol = 1e-5
iters = 100
dtype = torch.float16
device = 'cpu'

C_phi = construct_direction_codebook(phi_bits, dtype=dtype, device=device)
C_r = construct_magnitude_codebook(r_bits, k, tau, tol, iters)
pcdvq = PCDVQ(directions_codebook=C_phi, magnitudes_codebook=C_r)

In [208]:
C_phi

tensor([[ 0.3535,  0.3535, -0.3535, -0.3535,  0.3535,  0.3535,  0.3535,  0.3535],
        [-0.3535, -0.3535,  0.3535,  0.3535, -0.3535, -0.3535, -0.3535, -0.3535],
        [ 0.7070,  0.0000,  0.7070,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.7070,  0.0000, -0.7070,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.7070,  0.0000,  0.7070,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.0000, -0.7070,  0.0000, -0.7070,  0.0000,  0.0000,  0.0000,  0.0000],
        [-0.3535, -0.3535,  0.3535,  0.3535,  0.3535,  0.3535,  0.3535,  0.3535],
        [ 0.3535, -0.3535, -0.3535,  0.3535, -0.3535, -0.3535,  0.3535,  0.3535],
        [-0.3535,  0.3535,  0.3535, -0.3535, -0.3535, -0.3535,  0.3535,  0.3535],
        [ 0.3535, -0.3535, -0.3535,  0.3535,  0.3535,  0.3535, -0.3535, -0.3535],
        [-0.3535,  0.3535,  0.3535, -0.3535,  0.3535,  0.3535, -0.3535, -0.3535],
        [ 0.3535,  0.3535, -0.3535, -0.3535, -0.3535, -0.3535, -0.3535, -0.3535],
        [ 0.7070

In [209]:
C_r

tensor([0.9047, 0.0111, 6.5041, 7.0397])

In [113]:
weight_reshaped = reshape_pq_to_k(weight, k)
pcdvq_res = pcdvq.forward(weight_reshaped)
weight_reshaped_quant = pcdvq_res["x_q"]
weight_quant = reshape_k_to_pq(weight_reshaped_quant, p, q)
mse_val = F.mse_loss(weight.float(), weight_quant.float()).item()
mse_val

/tmp/ipykernel_240183/3219234353.py:5: UserWarning: Using a target size (torch.Size([163840, 64])) that is different to the input size (torch.Size([4096, 2560])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  mse_val = F.mse_loss(weight.float(), weight_quant.float()).item()


RuntimeError: The size of tensor a (2560) must match the size of tensor b (64) at non-singleton dimension 1

In [210]:
weight_quant

tensor([[-0.2339, -0.0823, -0.1465,  ...,  0.0735,  0.1311, -0.0461],
        [ 0.2339, -0.0823,  0.1465,  ..., -0.0735, -0.1311, -0.0461],
        [ 0.2339,  0.0823,  0.1465,  ...,  0.0735, -0.1311, -0.0461],
        ...,
        [ 0.2339,  0.0823, -0.1465,  ..., -0.0735, -0.1311, -0.0461],
        [ 0.2339,  0.0823,  0.1465,  ...,  0.0735, -0.1311, -0.0461],
        [ 0.2339,  0.0823,  0.1465,  ...,  0.0735,  0.1311,  0.0461]],
       dtype=torch.float16)

In [211]:
pcdvq_res

{'phis': tensor([[1.6074, 1.8340, 1.5166,  ..., 1.2754, 0.3440, 0.1016],
         [1.6572, 1.5518, 1.9570,  ..., 1.6650, 2.3594, 2.5156],
         [1.6221, 1.6406, 1.6484,  ..., 1.3447, 0.8242, 5.7500],
         ...,
         [1.6865, 1.5020, 1.7129,  ..., 1.9258, 0.3850, 2.9375],
         [1.4805, 1.5820, 1.7432,  ..., 1.0010, 1.6377, 1.7500],
         [1.5869, 1.4365, 1.5117,  ..., 0.7715, 1.5625, 2.5859]],
        dtype=torch.float16),
 'r': tensor([0.0606, 0.0674, 0.0710,  ..., 0.1722, 0.2096, 0.1864],
        dtype=torch.float16),
 'idx_dir': tensor([239, 239, 239,  ..., 239, 239, 239]),
 'idx_rad': tensor([0, 0, 0,  ..., 0, 0, 0]),
 'phis_q': tensor([[0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         ...,
         [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
         [0.5000, 0.5000, 0.5000,  ..., 0.5000, 0.5000, 0.5000],
     

In [212]:
prng = torch.Generator(device='cpu')
prng.manual_seed(42)
state = prng.get_state()
p, q = weight_reshaped.shape
weight_sgr = randomized_hadamard_transform(pad_to_power_of_2(weight_reshaped.T), prng=prng).T
weight_sgr, weight_sgr.shape, p, q

(tensor([[ 0.0007, -0.0225, -0.0005,  ..., -0.0354,  0.0405,  0.0073],
         [-0.0331,  0.0036,  0.0083,  ...,  0.0355, -0.0028, -0.0083],
         [-0.0019, -0.0018, -0.0326,  ...,  0.0063,  0.0162,  0.0103],
         ...,
         [-0.0055,  0.0359, -0.0056,  ...,  0.0054, -0.0235, -0.0157],
         [ 0.0046, -0.0269, -0.0325,  ..., -0.0015, -0.0049,  0.0026],
         [-0.0243,  0.0424, -0.0294,  ..., -0.0042,  0.0109, -0.0156]],
        dtype=torch.float16),
 torch.Size([262144, 64]),
 163840,
 64)

In [213]:
prng.set_state(state)
weight_sgr_reverse = inverse_randomized_hadamard_transform(weight_sgr.T, prng=prng).T[:p]
weight_sgr_reverse, weight_sgr_reverse.shape

(tensor([[-0.0002, -0.0102,  0.0095,  ..., -0.0073, -0.0130,  0.0064],
         [-0.0059, -0.0166, -0.0086,  ...,  0.0088,  0.0068, -0.0117],
         [ 0.0015, -0.0015, -0.0054,  ..., -0.0053,  0.0119,  0.0071],
         ...,
         [-0.0312, -0.0279, -0.0066,  ...,  0.0045,  0.0248, -0.0245],
         [-0.0398,  0.0214,  0.0175,  ...,  0.0432, -0.0012, -0.0088],
         [-0.0211,  0.0301, -0.0251,  ...,  0.0305, -0.0235, -0.0231]],
        dtype=torch.float16),
 torch.Size([163840, 64]))

In [214]:
weight_reshaped

tensor([[-0.0002, -0.0102,  0.0096,  ..., -0.0073, -0.0129,  0.0064],
        [-0.0059, -0.0166, -0.0086,  ...,  0.0088,  0.0068, -0.0117],
        [ 0.0015, -0.0015, -0.0055,  ..., -0.0053,  0.0119,  0.0070],
        ...,
        [-0.0312, -0.0280, -0.0066,  ...,  0.0045,  0.0248, -0.0245],
        [-0.0398,  0.0214,  0.0175,  ...,  0.0432, -0.0012, -0.0088],
        [-0.0211,  0.0302, -0.0250,  ...,  0.0305, -0.0234, -0.0231]],
       dtype=torch.float16)

In [215]:
assert torch.allclose(weight_reshaped, weight_sgr_reverse, atol=1e-3)

In [216]:
phi, r = PCDVQ.to_polar(weight_sgr)
reshaped_phis = reshape_pq_to_k(phi, 8)
reshaped_phis

tensor([[1.5654, 1.7227, 1.5742,  ..., 1.3711, 1.5000, 1.5205],
        [1.5752, 1.5547, 1.5586,  ..., 1.3525, 1.7334, 1.4756],
        [1.3525, 1.3516, 1.4395,  ..., 1.7480, 1.7324, 1.6973],
        ...,
        [1.6758, 1.6426, 1.9775,  ..., 1.8545, 1.5361, 1.5762],
        [1.6221, 1.8008, 1.6113,  ..., 1.6260, 2.2812, 1.4561],
        [1.6621, 2.3770, 1.1875,  ..., 2.0449, 1.7900, 5.3203]],
       dtype=torch.float16)

In [217]:
z = torch.nn.functional.normalize(reshaped_phis, dim=-1)
Z = torch.nn.functional.normalize(C_phi.to(z.dtype), dim=-1)
sim = z @ Z.T
idx_dir = sim.argmax(dim=1)
sim

tensor([[ 0.5005, -0.5005,  0.5068,  ..., -0.4670,  0.4875, -0.4875],
        [ 0.5039, -0.5039,  0.5059,  ..., -0.4565,  0.5181, -0.5181],
        [ 0.5195, -0.5195,  0.4504,  ..., -0.5557,  0.5532, -0.5532],
        ...,
        [ 0.4861, -0.4861,  0.5356,  ..., -0.5029,  0.4561, -0.4561],
        [ 0.4861, -0.4861,  0.4709,  ..., -0.4492,  0.5444, -0.5444],
        [ 0.6484, -0.6484,  0.2822,  ..., -0.7295,  0.7041, -0.7041]],
       dtype=torch.float16)

In [218]:
idx_dir

tensor([26, 54, 58,  ..., 38, 48, 32])

In [219]:
d = (r.view(-1, 1) - C_r.view(1, -1)).abs()
idx_rad = d.argmin(dim=1)
idx_rad

tensor([1, 1, 1,  ..., 1, 1, 1])

In [220]:
r

tensor([0.1486, 0.1501, 0.1587,  ..., 0.1617, 0.1320, 0.1486],
       dtype=torch.float16)

In [221]:
phis_q = Z[idx_dir]
phi_p, phi_q = phi.shape
reshaped_phis_q = reshape_k_to_pq(phis_q, phi_p, phi_q)
r_q = C_r[idx_rad].unsqueeze(1)
prng.set_state(state)
weight_q = inverse_randomized_hadamard_transform(PCDVQ.to_cartesian(reshaped_phis_q, r_q).T, prng=prng).T[:p]

In [222]:
weight_q

tensor([[-5.5400e+00,  3.9788e-01,  3.8280e-02,  ...,  0.0000e+00,
         -0.0000e+00, -0.0000e+00],
        [ 1.8272e-02, -1.3904e-01, -4.0908e-02,  ...,  0.0000e+00,
         -0.0000e+00, -0.0000e+00],
        [-1.0417e-02, -5.3465e-02,  2.1540e-02,  ..., -0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        ...,
        [-2.8018e-04, -1.7252e-03,  7.1873e-04,  ...,  0.0000e+00,
         -0.0000e+00,  0.0000e+00],
        [-7.6397e-04, -3.3922e-03,  4.5999e-05,  ...,  0.0000e+00,
          0.0000e+00, -0.0000e+00],
        [ 9.5730e-04, -2.6090e-03, -8.7808e-04,  ...,  0.0000e+00,
          0.0000e+00, -0.0000e+00]])

In [223]:
phis_q, r_q

(tensor([[0.0000, 0.7070, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.7070, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.7070, 0.7070, 0.0000],
         ...,
         [0.0000, 0.0000, 0.7070,  ..., 0.7070, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.7070, 0.0000],
         [0.0000, 0.7070, 0.0000,  ..., 0.0000, 0.0000, 0.7070]],
        dtype=torch.float16),
 tensor([[0.0111],
         [0.0111],
         [0.0111],
         ...,
         [0.0111],
         [0.0111],
         [0.0111]]))

In [179]:
phis_q[5]

tensor([0., 1., 0., 0., 0., 1., 0., 0.], dtype=torch.float16)

In [226]:
# C — кодбук (K,8), qdir — квантизированные направления (...,8), idx — индексы (из квантизатора)

# 1) Проверка соответствия: qdir == C[idx]
ok = (C_phi[idx_dir.reshape(-1)] - phis_q.reshape(-1, 8)).abs().max().item() < 1e-6
print("qdir equals codebook rows:", ok)

# 2) Есть ли отрицательные значения и тип-B (все 8 компонент ≈ 0.3536 по модулю)?
print("qdir has negatives:", (phis_q < 0).any().item())
abs_vals = torch.unique(phis_q.abs().flatten().round(decimals=4))
print("unique |values| in qdir:", abs_vals.tolist())

# тип-B присутствует, если есть блоки, где все 8 компонент по модулю ~0.3536
typeB_in_qdir = (((phis_q.abs() > 0.30) & (phis_q.abs() < 0.40)).sum(dim=-1) == 8).any().item()
print("type-B present in qdir:", typeB_in_qdir)


qdir equals codebook rows: True
qdir has negatives: True
unique |values| in qdir: [0.0, 0.353515625, 0.70703125]
type-B present in qdir: True


In [227]:
x = reshaped_phis
G = x.size(-1)//8
r = torch.linalg.vector_norm(x.reshape(-1, G, 8).float(), dim=-1)
print("r min/median/max:", r.min().item(), r.median().item(), r.max().item())


r min/median/max: 2.576680898666382 4.491796016693115 8.53427505493164


In [228]:
xg = x.reshape(-1, G, 8).float()
u  = xg / (torch.linalg.vector_norm(xg, dim=-1, keepdim=True).clamp_min(1e-12))
cos_mean = (u.reshape(-1,8) * phis_q.reshape(-1,8)).sum(-1).mean().item()
print("mean cosine(u, qdir):", round(cos_mean, 4))


mean cosine(u, qdir): 0.5858
